In [9]:
import torch
from torch import nn
import torchvision
from torchvision import transforms

In [10]:
class FashionMNIST():
    def __init__(self, batch_size=64, resize=(28, 28)):
        self.batch_size = batch_size
        self.resize = resize
        self.transform = transforms.Compose([transforms.Resize(self.resize),
                                             transforms.ToTensor()])
        self.train = torchvision.datasets.FashionMNIST(root='/home/beret/Documents/moje_projekty/pytorch_study/4_Linear_classification/root', train=True, download=True,
                                                       transform=self.transform)
        self.validation = torchvision.datasets.FashionMNIST(root='/home/beret/Documents/moje_projekty/pytorch_study/4_Linear_classification/root', train=False, download=True,
                                                            transform=self.transform)

    def get_dataloader(self, train):
        data = self.train if train else self.validation
        dataloader = torch.utils.data.DataLoader(data, batch_size=self.batch_size, shuffle=train)
        return dataloader

    def training_data(self):
        return self.get_dataloader(True)

    def validation_data(self):
        return self.get_dataloader(False)

In [11]:
def init_cnn(module):
    if isinstance(module, (nn.Linear, nn.Conv2d, nn.LazyLinear, nn.LazyConv2d)):
        nn.init.xavier_uniform_(module.weight)

class LeNet(nn.Module):
    def __init__(self, lr, num_classes):
        super().__init__()
        self.lr= lr
        self.loss_function = nn.CrossEntropyLoss()
        self.net = nn.Sequential(nn.LazyConv2d(6, kernel_size=5, padding=2), nn.Sigmoid(), nn.AvgPool2d(kernel_size=2, stride=2),nn.LazyConv2d(16, kernel_size=5), nn.Sigmoid(), nn.AvgPool2d(kernel_size=2, stride=2),nn.Flatten(), nn.LazyLinear(120), nn.Sigmoid(), nn.LazyLinear(84), nn.Sigmoid(), nn.LazyLinear(num_classes))

    def forward(self, X):
        return self.net(X)

    def loss(self, y_hat, y):
        return self.loss_function(y_hat, y)

    def get_sgd_optimizer(self, params, weight_decay):
        return torch.optim.SGD(params, lr=self.lr, weight_decay=weight_decay)



In [12]:
def Train_batches(dataloader, model, optimizer, device):

    model.train()
    for (X, y) in dataloader:
        X,y = X.to(device), y.to(device)

        pred = model(X)
        loss = model.loss(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



def validate_batches(dataloader, model, device):

    total_loss, correct = 0, 0
    model.eval()
    for (X, y) in dataloader:

        X,y = X.to(device), y.to(device)

        with torch.no_grad():
            y_hat = model(X)
            total_loss += model.loss(y_hat, y).item()
            correct += (y_hat.argmax(dim=1) == y).float().sum().item()

    total_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    print(f"validation loss: {total_loss :> 8f} and correct: {(100 * correct) :>0.1f}%")

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = FashionMNIST(batch_size=256)
data_train = data.training_data()
data_validation = data.validation_data()
model = LeNet(lr=0.1, num_classes=10).to(device)
# 3. FIX: Dummy forward pass required to initialize Lazy modules
dummy_input = torch.randn(1, 1, 28, 28).to(device)
model(dummy_input)

# 4. Now we can apply weights and set up the optimizer
model.apply(init_cnn)
optim = model.get_sgd_optimizer(model.parameters(),weight_decay=1e-12)



for i in range(10):
    Train_batches(data_train, model, optim, device)
    validate_batches(data_validation, model, device)


validation loss:  2.304125 and correct: 10.0%
validation loss:  2.307988 and correct: 10.0%
validation loss:  2.304249 and correct: 10.0%
validation loss:  2.310187 and correct: 10.0%
validation loss:  2.307907 and correct: 10.0%
validation loss:  2.298671 and correct: 10.0%
validation loss:  2.234108 and correct: 31.8%
validation loss:  1.707263 and correct: 39.8%
validation loss:  1.291232 and correct: 53.9%
validation loss:  1.140388 and correct: 56.8%
